# Account Takeover (ATO) Fraud Detection in Payments & Lending using KumoRFM

<a target="_blank" href="https://colab.research.google.com/github/AbhinavKhareTech/kumo-rfm/blob/contrib/ato-fraud-detection-notebook/notebooks/ato_fraud_detection.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Series:** Advanced Relational Fraud Detection with KumoRFM (Module 2 of N)
- Module 1: [Merchant Collusion Ring Detection](bfsi_fraud_detection.ipynb)
- **Module 2: Account Takeover Fraud Detection** (this notebook)

---

## Overview

Account Takeover (ATO) occurs when attackers gain control of legitimate user accounts through credential stuffing, phishing, or session hijacking, and then execute rapid fraudulent activity: high-velocity transactions, loan applications, or cash-outs.

**Why traditional systems struggle with ATO:**

Traditional fraud systems rely on per-transaction rules and flat ML features (velocity counters, device fingerprints, geo-anomalies). Sophisticated attackers defeat these by mimicking legitimate behavior patterns. The real signal in ATO is **structural and behavioral**: it lives in the *relationships* between users, devices, sessions, transactions, and loan applications over time.

**What KumoRFM brings:**

KumoRFM's graph transformer architecture natively reasons over multi-table relational data, capturing behavioral deviations and structural anomalies that signal account compromise, without manual feature engineering.

### What this notebook covers

1. Synthetic multi-table dataset generation with injected ATO attack patterns
2. Relational graph construction using `LocalGraph.from_data()` with temporal relationships
3. PQL-based real-time ATO risk scoring via missing-value imputation
4. Head-to-head comparison against a strong XGBoost baseline
5. Takeover graph visualization and behavioral change detection
6. Production deployment considerations

### Requirements

- Python 3.9+
- A valid [KumoRFM API key](https://kumorfm.ai) (free tier available)
- Libraries: `kumoai`, `pandas`, `numpy`, `scikit-learn`, `xgboost`, `networkx`, `matplotlib`, `seaborn`

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install -q kumoai xgboost networkx matplotlib seaborn scikit-learn pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

print("Environment ready.")

## 2. Synthetic Dataset Generation

We generate a realistic multi-table dataset that models the core entities in a payments and lending platform. The schema has five interconnected tables:

| Table | Role | Key Relationships |
|---|---|---|
| `users` | Account holders | Primary entity |
| `devices` | Physical/virtual devices | Many-to-many with users via sessions |
| `sessions` | Login events | Links users to devices with temporal context |
| `transactions` | Payment events | Foreign key to users, temporal |
| `loan_applications` | Lending events | Foreign key to users, temporal |

### Injected ATO Attack Patterns

We embed three realistic ATO variants into the synthetic data:

1. **Rapid Takeover**: Attacker gains access and immediately executes high-value transactions and loan applications from a new device within hours.
2. **Slow-Burn Takeover**: Attacker operates the account gradually over days, slowly escalating transaction amounts to avoid velocity triggers.
3. **Cross-Product ATO**: Attacker uses a compromised account to apply for loans while simultaneously draining the payments balance.

Each pattern creates **structural anomalies** in the relational graph (new device-user edges, unusual session-transaction temporal patterns) that flat features cannot capture effectively.

In [ ]:
# ==============================================================
# 2.1 Generate base entity tables
# ==============================================================

N_USERS = 500
N_DEVICES = 300
N_LEGITIMATE_SESSIONS = 4000
N_LEGITIMATE_TXNS = 8000
N_LEGITIMATE_LOANS = 600

# --- Users table ---
user_ids = [f"U{i:04d}" for i in range(N_USERS)]
cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad",
          "Pune", "Kolkata", "Ahmedabad", "Jaipur", "Lucknow"]

users_df = pd.DataFrame({
    "user_id": user_ids,
    "signup_date": pd.date_range("2023-01-01", periods=N_USERS, freq="4h"),
    "city": np.random.choice(cities, N_USERS),
    "account_type": np.random.choice(["savings", "current", "premium"], N_USERS,
                                      p=[0.5, 0.35, 0.15]),
    "kyc_verified": np.random.choice([True, False], N_USERS, p=[0.85, 0.15]),
})

# --- Devices table ---
device_ids = [f"D{i:04d}" for i in range(N_DEVICES)]
device_types = ["mobile_android", "mobile_ios", "desktop_windows",
                "desktop_mac", "tablet"]

devices_df = pd.DataFrame({
    "device_id": device_ids,
    "device_type": np.random.choice(device_types, N_DEVICES,
                                     p=[0.35, 0.25, 0.2, 0.1, 0.1]),
    "os_version": [f"v{np.random.randint(10, 18)}.{np.random.randint(0, 9)}"
                   for _ in range(N_DEVICES)],
    "first_seen": pd.date_range("2023-01-01", periods=N_DEVICES, freq="6h"),
})

print(f"Users: {len(users_df)}, Devices: {len(devices_df)}")
users_df.head()

In [ ]:
# ==============================================================
# 2.2 Generate legitimate sessions
# ==============================================================

base_date = pd.Timestamp("2023-06-01")

# Each user has a small set of "owned" devices (1-3)
user_device_map = {}
for uid in user_ids:
    n_devices = np.random.choice([1, 2, 3], p=[0.5, 0.35, 0.15])
    user_device_map[uid] = list(np.random.choice(device_ids, n_devices, replace=False))

session_records = []
for i in range(N_LEGITIMATE_SESSIONS):
    uid = np.random.choice(user_ids)
    did = np.random.choice(user_device_map[uid])
    ts = base_date + pd.Timedelta(hours=np.random.randint(0, 4320))  # ~6 months
    duration = np.random.exponential(15) + 1  # minutes
    session_records.append({
        "session_id": f"S{i:06d}",
        "user_id": uid,
        "device_id": did,
        "login_time": ts,
        "duration_minutes": round(duration, 1),
        "login_method": np.random.choice(
            ["password", "biometric", "otp", "sso"],
            p=[0.4, 0.3, 0.2, 0.1]
        ),
        "ip_country": "IN",
        "is_ato_session": False,
    })

sessions_df = pd.DataFrame(session_records)
print(f"Legitimate sessions: {len(sessions_df)}")

In [ ]:
# ==============================================================
# 2.3 Generate legitimate transactions
# ==============================================================

txn_records = []
for i in range(N_LEGITIMATE_TXNS):
    uid = np.random.choice(user_ids)
    ts = base_date + pd.Timedelta(hours=np.random.randint(0, 4320))
    amount = np.random.lognormal(mean=6.0, sigma=1.2)  # median ~400 INR
    txn_records.append({
        "txn_id": f"T{i:06d}",
        "user_id": uid,
        "txn_time": ts,
        "amount": round(min(amount, 50000), 2),
        "txn_type": np.random.choice(
            ["upi", "neft", "card", "wallet"],
            p=[0.45, 0.2, 0.25, 0.1]
        ),
        "merchant_category": np.random.choice(
            ["grocery", "electronics", "travel", "food", "utilities", "transfer"],
            p=[0.2, 0.15, 0.1, 0.25, 0.15, 0.15]
        ),
        "is_ato_txn": False,
    })

transactions_df = pd.DataFrame(txn_records)
print(f"Legitimate transactions: {len(transactions_df)}")

In [ ]:
# ==============================================================
# 2.4 Generate legitimate loan applications
# ==============================================================

loan_records = []
for i in range(N_LEGITIMATE_LOANS):
    uid = np.random.choice(user_ids)
    ts = base_date + pd.Timedelta(hours=np.random.randint(0, 4320))
    amount = np.random.choice([10000, 25000, 50000, 100000, 200000, 500000],
                               p=[0.3, 0.25, 0.2, 0.15, 0.07, 0.03])
    loan_records.append({
        "loan_app_id": f"L{i:05d}",
        "user_id": uid,
        "application_time": ts,
        "loan_amount": amount,
        "loan_type": np.random.choice(
            ["personal", "credit_line", "bnpl", "education"],
            p=[0.3, 0.3, 0.3, 0.1]
        ),
        "tenure_months": np.random.choice([3, 6, 12, 24, 36]),
        "is_ato_loan": False,
    })

loans_df = pd.DataFrame(loan_records)
print(f"Legitimate loan applications: {len(loans_df)}")

### 2.5 Injecting ATO Attack Patterns

We now inject three distinct ATO variants. Each compromised user gets sessions, transactions, and/or loans from **attacker-controlled devices** that have no prior association with the user, creating structural graph anomalies.

In [ ]:
# ==============================================================
# 2.5 Inject ATO attack patterns
# ==============================================================

ATO_START = pd.Timestamp("2023-10-01")
N_ATO_USERS = 40  # ~8% of users compromised

ato_user_ids = list(np.random.choice(user_ids, N_ATO_USERS, replace=False))

# Attacker devices (never seen before by these users)
attacker_device_pool = list(np.random.choice(
    [d for d in device_ids if all(d not in user_device_map[u] for u in ato_user_ids[:10])],
    20, replace=False
))

ato_sessions = []
ato_txns = []
ato_loans = []
sid_counter = N_LEGITIMATE_SESSIONS
tid_counter = N_LEGITIMATE_TXNS
lid_counter = N_LEGITIMATE_LOANS

for idx, uid in enumerate(ato_user_ids):
    attack_device = np.random.choice(attacker_device_pool)

    if idx < 15:
        # --- Pattern 1: Rapid Takeover ---
        # Burst of activity within 2-6 hours from a new device
        burst_start = ATO_START + pd.Timedelta(days=np.random.randint(0, 30))
        for j in range(np.random.randint(3, 7)):
            ts = burst_start + pd.Timedelta(minutes=np.random.randint(5, 360))
            ato_sessions.append({
                "session_id": f"S{sid_counter:06d}",
                "user_id": uid,
                "device_id": attack_device,
                "login_time": ts,
                "duration_minutes": round(np.random.uniform(0.5, 3.0), 1),
                "login_method": np.random.choice(["password", "otp"]),
                "ip_country": np.random.choice(["IN", "NG", "RU", "CN"], p=[0.3, 0.3, 0.2, 0.2]),
                "is_ato_session": True,
            })
            sid_counter += 1

            # High-value transaction per session
            ato_txns.append({
                "txn_id": f"T{tid_counter:06d}",
                "user_id": uid,
                "txn_time": ts + pd.Timedelta(minutes=np.random.randint(1, 10)),
                "amount": round(np.random.uniform(15000, 49999), 2),
                "txn_type": np.random.choice(["upi", "neft"]),
                "merchant_category": "transfer",
                "is_ato_txn": True,
            })
            tid_counter += 1

    elif idx < 28:
        # --- Pattern 2: Slow-Burn Takeover ---
        # Gradual escalation over 7-14 days
        slow_start = ATO_START + pd.Timedelta(days=np.random.randint(0, 20))
        for day_offset in range(np.random.randint(5, 14)):
            ts = slow_start + pd.Timedelta(days=day_offset,
                                           hours=np.random.randint(9, 22))
            ato_sessions.append({
                "session_id": f"S{sid_counter:06d}",
                "user_id": uid,
                "device_id": attack_device,
                "login_time": ts,
                "duration_minutes": round(np.random.uniform(5, 20), 1),
                "login_method": "password",
                "ip_country": "IN",
                "is_ato_session": True,
            })
            sid_counter += 1

            # Escalating amounts
            base_amt = 500 * (1 + day_offset * 0.8)
            ato_txns.append({
                "txn_id": f"T{tid_counter:06d}",
                "user_id": uid,
                "txn_time": ts + pd.Timedelta(minutes=np.random.randint(2, 30)),
                "amount": round(base_amt + np.random.uniform(0, 500), 2),
                "txn_type": "upi",
                "merchant_category": np.random.choice(["transfer", "electronics"]),
                "is_ato_txn": True,
            })
            tid_counter += 1

    else:
        # --- Pattern 3: Cross-Product ATO ---
        # Simultaneous transactions + loan applications
        cross_start = ATO_START + pd.Timedelta(days=np.random.randint(0, 25))
        for j in range(np.random.randint(2, 5)):
            ts = cross_start + pd.Timedelta(hours=np.random.randint(1, 48))
            ato_sessions.append({
                "session_id": f"S{sid_counter:06d}",
                "user_id": uid,
                "device_id": attack_device,
                "login_time": ts,
                "duration_minutes": round(np.random.uniform(1, 8), 1),
                "login_method": "password",
                "ip_country": np.random.choice(["IN", "NG", "RU"]),
                "is_ato_session": True,
            })
            sid_counter += 1

            # Transaction
            ato_txns.append({
                "txn_id": f"T{tid_counter:06d}",
                "user_id": uid,
                "txn_time": ts + pd.Timedelta(minutes=5),
                "amount": round(np.random.uniform(10000, 40000), 2),
                "txn_type": "neft",
                "merchant_category": "transfer",
                "is_ato_txn": True,
            })
            tid_counter += 1

            # Loan application
            ato_loans.append({
                "loan_app_id": f"L{lid_counter:05d}",
                "user_id": uid,
                "application_time": ts + pd.Timedelta(minutes=np.random.randint(10, 60)),
                "loan_amount": np.random.choice([100000, 200000, 500000]),
                "loan_type": np.random.choice(["personal", "credit_line"]),
                "tenure_months": np.random.choice([3, 6]),
                "is_ato_loan": True,
            })
            lid_counter += 1

# Merge ATO data into main tables
sessions_df = pd.concat([sessions_df, pd.DataFrame(ato_sessions)], ignore_index=True)
transactions_df = pd.concat([transactions_df, pd.DataFrame(ato_txns)], ignore_index=True)
loans_df = pd.concat([loans_df, pd.DataFrame(ato_loans)], ignore_index=True)

print(f"Total sessions: {len(sessions_df)} "
      f"(ATO: {sessions_df['is_ato_session'].sum()})")
print(f"Total transactions: {len(transactions_df)} "
      f"(ATO: {transactions_df['is_ato_txn'].sum()})")
print(f"Total loan applications: {len(loans_df)} "
      f"(ATO: {loans_df['is_ato_loan'].sum()})")

In [ ]:
# ==============================================================
# 2.6 Create user-level ATO labels for evaluation
# ==============================================================

# A user is labeled ATO-positive if they have any ATO session/txn/loan
ato_users_set = set(ato_user_ids)
users_df["is_ato_victim"] = users_df["user_id"].isin(ato_users_set).astype(int)

print(f"ATO victims: {users_df['is_ato_victim'].sum()} / {len(users_df)} "
      f"({users_df['is_ato_victim'].mean()*100:.1f}%)")
users_df.head()

## 3. Exploratory Data Analysis

Before building models, we examine the structural and behavioral differences between legitimate and ATO-compromised activity.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 3a: Transaction amount distributions
legit_txns = transactions_df[~transactions_df["is_ato_txn"]]
ato_txns_only = transactions_df[transactions_df["is_ato_txn"]]

axes[0].hist(legit_txns["amount"].clip(upper=30000), bins=50, alpha=0.6,
             label="Legitimate", color="steelblue", density=True)
axes[0].hist(ato_txns_only["amount"].clip(upper=30000), bins=30, alpha=0.6,
             label="ATO", color="crimson", density=True)
axes[0].set_title("Transaction Amount Distribution")
axes[0].set_xlabel("Amount (INR)")
axes[0].legend()

# 3b: Session duration
legit_sess = sessions_df[~sessions_df["is_ato_session"]]
ato_sess = sessions_df[sessions_df["is_ato_session"]]

axes[1].hist(legit_sess["duration_minutes"].clip(upper=60), bins=40, alpha=0.6,
             label="Legitimate", color="steelblue", density=True)
axes[1].hist(ato_sess["duration_minutes"].clip(upper=60), bins=20, alpha=0.6,
             label="ATO", color="crimson", density=True)
axes[1].set_title("Session Duration (minutes)")
axes[1].set_xlabel("Duration")
axes[1].legend()

# 3c: Devices per user
devices_per_user = sessions_df.groupby("user_id")["device_id"].nunique()
ato_dpf = devices_per_user[devices_per_user.index.isin(ato_users_set)]
legit_dpf = devices_per_user[~devices_per_user.index.isin(ato_users_set)]

axes[2].bar(["Legitimate", "ATO"],
            [legit_dpf.mean(), ato_dpf.mean()],
            color=["steelblue", "crimson"], alpha=0.7)
axes[2].set_title("Avg Unique Devices per User")
axes[2].set_ylabel("Unique Devices")

plt.tight_layout()
plt.show()

print(f"Avg transaction amount - Legitimate: {legit_txns['amount'].mean():.0f} INR, "
      f"ATO: {ato_txns_only['amount'].mean():.0f} INR")
print(f"Avg session duration - Legitimate: {legit_sess['duration_minutes'].mean():.1f} min, "
      f"ATO: {ato_sess['duration_minutes'].mean():.1f} min")

## 4. Relational Graph Construction with KumoRFM

This is where KumoRFM's power becomes evident. Instead of flattening the five tables into a single feature matrix (which loses relational structure), we construct a **temporal heterogeneous graph** that preserves all entity relationships.

### Graph Schema

```
users ---- sessions ---- devices
  |
  +------- transactions
  |
  +------- loan_applications
```

KumoRFM's `LocalGraph.from_data()` automatically infers primary keys, foreign keys, time columns, and semantic types. We then verify and adjust the inferred metadata.

In [ ]:
import kumoai.experimental.rfm as rfm
import os

# NOTE: Replace with your API key or set KUMO_API_KEY environment variable
os.environ["KUMO_API_KEY"] = os.environ.get("KUMO_API_KEY", "ENTER_YOUR_API_KEY_HERE")
rfm.init()

In [ ]:
# ==============================================================
# 4.1 Prepare DataFrames for graph construction
# ==============================================================

# For KumoRFM, we prepare clean DataFrames with the target column.
# The target (is_ato_victim) will have labels masked for test users
# to enable missing-value imputation.

TRAIN_CUTOFF = pd.Timestamp("2023-09-15")

# Create the users table with the ATO label
users_graph_df = users_df[["user_id", "signup_date", "city",
                            "account_type", "kyc_verified", "is_ato_victim"]].copy()

# Temporal train/test split: mask labels for users whose ATO
# activity (if any) occurs after the cutoff
# For non-ATO users in test set, also mask to create prediction targets
test_user_mask = users_graph_df["user_id"].isin(
    # Users with late ATO activity + random sample of clean users for test
    set(ato_user_ids) | set(np.random.choice(
        [u for u in user_ids if u not in ato_users_set],
        size=100, replace=False
    ))
)
test_user_ids = set(users_graph_df.loc[test_user_mask, "user_id"])

# Store ground truth before masking
ground_truth = users_graph_df[["user_id", "is_ato_victim"]].copy()
ground_truth = ground_truth[ground_truth["user_id"].isin(test_user_ids)]

# Mask test labels for KumoRFM imputation
users_graph_df.loc[test_user_mask, "is_ato_victim"] = pd.NA

# Prepare session/txn/loan tables (drop the per-row fraud labels)
sessions_graph_df = sessions_df[["session_id", "user_id", "device_id",
                                  "login_time", "duration_minutes",
                                  "login_method", "ip_country"]].copy()

transactions_graph_df = transactions_df[["txn_id", "user_id", "txn_time",
                                          "amount", "txn_type",
                                          "merchant_category"]].copy()

loans_graph_df = loans_df[["loan_app_id", "user_id", "application_time",
                            "loan_amount", "loan_type", "tenure_months"]].copy()

devices_graph_df = devices_df[["device_id", "device_type", "os_version",
                                "first_seen"]].copy()

print(f"Test users: {len(test_user_ids)} "
      f"(ATO: {ground_truth['is_ato_victim'].sum()}, "
      f"Clean: {(ground_truth['is_ato_victim']==0).sum()})")

In [ ]:
# ==============================================================
# 4.2 Build the relational graph
# ==============================================================

graph = rfm.LocalGraph.from_data({
    "users": users_graph_df,
    "devices": devices_graph_df,
    "sessions": sessions_graph_df,
    "transactions": transactions_graph_df,
    "loan_applications": loans_graph_df,
})

print("Graph constructed. Reviewing inferred metadata...")

In [ ]:
# ==============================================================
# 4.3 Verify and adjust graph metadata
# ==============================================================

# Ensure time columns are correctly set
graph["sessions"].time_col = "login_time"
graph["transactions"].time_col = "txn_time"
graph["loan_applications"].time_col = "application_time"
graph["users"].time_col = "signup_date"
graph["devices"].time_col = "first_seen"

# Ensure semantic types
graph["transactions"]["amount"].stype = "numerical"
graph["loan_applications"]["loan_amount"].stype = "numerical"
graph["loan_applications"]["tenure_months"].stype = "numerical"
graph["sessions"]["duration_minutes"].stype = "numerical"

graph["users"]["city"].stype = "categorical"
graph["users"]["account_type"].stype = "categorical"
graph["users"]["kyc_verified"].stype = "categorical"
graph["sessions"]["login_method"].stype = "categorical"
graph["sessions"]["ip_country"].stype = "categorical"
graph["transactions"]["txn_type"].stype = "categorical"
graph["transactions"]["merchant_category"].stype = "categorical"
graph["loan_applications"]["loan_type"].stype = "categorical"
graph["devices"]["device_type"].stype = "categorical"

# Ensure primary keys
graph["users"]["user_id"].stype = "ID"
graph["devices"]["device_id"].stype = "ID"
graph["sessions"]["session_id"].stype = "ID"
graph["transactions"]["txn_id"].stype = "ID"
graph["loan_applications"]["loan_app_id"].stype = "ID"

# Ensure the target column type
graph["users"]["is_ato_victim"].stype = "numerical"

# Verify edges (add if not auto-inferred)
try:
    graph.add_edge("sessions.user_id", "users.user_id")
except Exception:
    pass  # Already inferred
try:
    graph.add_edge("sessions.device_id", "devices.device_id")
except Exception:
    pass
try:
    graph.add_edge("transactions.user_id", "users.user_id")
except Exception:
    pass
try:
    graph.add_edge("loan_applications.user_id", "users.user_id")
except Exception:
    pass

print("Graph metadata verified and adjusted.")
print(graph)

## 5. ATO Risk Scoring with KumoRFM

We use KumoRFM's **missing-value imputation** to predict the `is_ato_victim` label for test users. The PQL query asks KumoRFM to fill in the masked labels by reasoning over the full relational graph structure.

This is the key advantage: KumoRFM does not need hand-engineered features. It directly learns from the multi-hop relationships (user <-> session <-> device, user <-> transactions, user <-> loans) to detect structural anomalies indicative of ATO.

**Note:** The cells below require a valid KumoRFM API key to execute.

In [ ]:
# ==============================================================
# 5.1 Initialize KumoRFM and run predictions
# ==============================================================

model = rfm.KumoRFM(graph)

# PQL query: predict is_ato_victim for test users via imputation
test_user_list = sorted(list(test_user_ids))
test_user_tuple = ", ".join([f"'{u}'" for u in test_user_list])

# For large sets, predict in batches
BATCH_SIZE = 50
kumo_predictions = []

for batch_start in range(0, len(test_user_list), BATCH_SIZE):
    batch = test_user_list[batch_start:batch_start + BATCH_SIZE]
    id_tuple = ", ".join([f"'{u}'" for u in batch])
    query = f"PREDICT users.is_ato_victim = 1 FOR users.user_id IN ({id_tuple})"

    try:
        result = model.predict(query)
        kumo_predictions.append(result)
        print(f"Batch {batch_start//BATCH_SIZE + 1}: "
              f"predicted {len(result)} users")
    except Exception as e:
        print(f"Batch {batch_start//BATCH_SIZE + 1} error: {e}")

if kumo_predictions:
    kumo_results_df = pd.concat(kumo_predictions, ignore_index=True)
    print(f"\nTotal KumoRFM predictions: {len(kumo_results_df)}")
    kumo_results_df.head(10)
else:
    print("No predictions returned. Check API key and graph setup.")
    kumo_results_df = None

In [ ]:
# ==============================================================
# 5.2 Evaluate KumoRFM predictions
# ==============================================================

from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    f1_score, classification_report
)

if kumo_results_df is not None:
    # Merge predictions with ground truth
    eval_df = ground_truth.merge(kumo_results_df, on="user_id", how="inner")

    # The prediction column name may vary; find it
    pred_cols = [c for c in eval_df.columns
                 if c not in ["user_id", "is_ato_victim"]]
    pred_col = pred_cols[0] if pred_cols else None

    if pred_col:
        y_true = eval_df["is_ato_victim"].values
        y_score = eval_df[pred_col].values

        roc_auc = roc_auc_score(y_true, y_score)
        pr_auc = average_precision_score(y_true, y_score)

        # F1 at 95% precision operating point
        prec_arr, rec_arr, thresholds = precision_recall_curve(y_true, y_score)
        f1_at_95 = 0.0
        for p, r, t in zip(prec_arr, rec_arr, thresholds):
            if p >= 0.95 and r > 0:
                f1 = 2 * p * r / (p + r)
                f1_at_95 = max(f1_at_95, f1)

        print("=" * 50)
        print("KumoRFM ATO Detection Results")
        print("=" * 50)
        print(f"ROC-AUC:          {roc_auc:.4f}")
        print(f"PR-AUC:           {pr_auc:.4f}")
        print(f"F1 @ 95% Prec:    {f1_at_95:.4f}")

        kumo_metrics = {"ROC-AUC": roc_auc, "PR-AUC": pr_auc,
                        "F1@95%Prec": f1_at_95}
    else:
        print("Could not identify prediction column.")
        kumo_metrics = None
else:
    print("Skipping evaluation (no KumoRFM predictions available).")
    kumo_metrics = None

## 6. XGBoost Baseline with Hand-Engineered Features

To benchmark KumoRFM, we build a strong XGBoost model using carefully engineered flat features. These represent the kind of features a production fraud team would typically construct:

**Velocity features:** Transaction count/sum in last 7/30 days, session frequency
**Device features:** Number of unique devices, new device indicator, device-sharing score
**Behavioral features:** Average session duration, login method distribution, IP country diversity
**Loan features:** Recent loan application count, total requested amount

This is the traditional approach: flatten the relational data into a single row per user, losing structural information in the process.

In [ ]:
# ==============================================================
# 6.1 Engineer flat features for XGBoost
# ==============================================================

def build_flat_features(users, sessions, transactions, loans, cutoff=None):
    """Build hand-engineered features per user."""
    features = users[["user_id"]].copy()

    # --- Transaction velocity features ---
    if cutoff:
        txn_window = transactions[transactions["txn_time"] <= cutoff]
    else:
        txn_window = transactions

    txn_agg = txn_window.groupby("user_id").agg(
        txn_count=("txn_id", "count"),
        txn_sum=("amount", "sum"),
        txn_mean=("amount", "mean"),
        txn_max=("amount", "max"),
        txn_std=("amount", "std"),
        n_txn_types=("txn_type", "nunique"),
        n_merchant_cats=("merchant_category", "nunique"),
    ).reset_index()

    # Recent velocity (last 7 days of data)
    if cutoff:
        recent_cutoff = cutoff - pd.Timedelta(days=7)
        recent_txns = txn_window[txn_window["txn_time"] >= recent_cutoff]
    else:
        latest = txn_window["txn_time"].max()
        recent_txns = txn_window[txn_window["txn_time"] >= latest - pd.Timedelta(days=7)]

    recent_agg = recent_txns.groupby("user_id").agg(
        recent_txn_count=("txn_id", "count"),
        recent_txn_sum=("amount", "sum"),
    ).reset_index()

    # --- Session/device features ---
    if cutoff:
        sess_window = sessions[sessions["login_time"] <= cutoff]
    else:
        sess_window = sessions

    sess_agg = sess_window.groupby("user_id").agg(
        session_count=("session_id", "count"),
        n_devices=("device_id", "nunique"),
        avg_duration=("duration_minutes", "mean"),
        n_login_methods=("login_method", "nunique"),
        n_ip_countries=("ip_country", "nunique"),
    ).reset_index()

    # New device indicator: device used in last 7 days not seen before
    if cutoff:
        recent_sess = sess_window[sess_window["login_time"] >= cutoff - pd.Timedelta(days=7)]
        old_sess = sess_window[sess_window["login_time"] < cutoff - pd.Timedelta(days=7)]
    else:
        latest_s = sess_window["login_time"].max()
        recent_sess = sess_window[sess_window["login_time"] >= latest_s - pd.Timedelta(days=7)]
        old_sess = sess_window[sess_window["login_time"] < latest_s - pd.Timedelta(days=7)]

    old_devices = old_sess.groupby("user_id")["device_id"].apply(set).to_dict()
    new_device_flags = []
    for uid in users["user_id"]:
        user_recent = recent_sess[recent_sess["user_id"] == uid]["device_id"].unique()
        user_old = old_devices.get(uid, set())
        has_new = int(any(d not in user_old for d in user_recent)) if len(user_recent) > 0 else 0
        new_device_flags.append({"user_id": uid, "has_new_device": has_new})
    new_device_df = pd.DataFrame(new_device_flags)

    # --- Loan features ---
    if cutoff:
        loan_window = loans[loans["application_time"] <= cutoff]
    else:
        loan_window = loans

    loan_agg = loan_window.groupby("user_id").agg(
        loan_app_count=("loan_app_id", "count"),
        total_loan_requested=("loan_amount", "sum"),
        max_loan_requested=("loan_amount", "max"),
    ).reset_index()

    # --- Merge all features ---
    features = features.merge(txn_agg, on="user_id", how="left")
    features = features.merge(recent_agg, on="user_id", how="left")
    features = features.merge(sess_agg, on="user_id", how="left")
    features = features.merge(new_device_df, on="user_id", how="left")
    features = features.merge(loan_agg, on="user_id", how="left")

    # Add user-level attributes
    features = features.merge(
        users[["user_id", "account_type", "kyc_verified"]],
        on="user_id", how="left"
    )

    # Encode categoricals
    features["account_type"] = features["account_type"].map(
        {"savings": 0, "current": 1, "premium": 2}
    ).fillna(0)
    features["kyc_verified"] = features["kyc_verified"].astype(int)

    features = features.fillna(0)
    return features

flat_features = build_flat_features(
    users_df, sessions_df, transactions_df, loans_df
)

print(f"Feature matrix shape: {flat_features.shape}")
print(f"Features: {[c for c in flat_features.columns if c != 'user_id']}")

In [ ]:
# ==============================================================
# 6.2 Train and evaluate XGBoost
# ==============================================================

import xgboost as xgb

# Prepare train/test split matching KumoRFM
train_mask = ~flat_features["user_id"].isin(test_user_ids)
test_mask = flat_features["user_id"].isin(test_user_ids)

feature_cols = [c for c in flat_features.columns
                if c not in ["user_id"]]

X_train = flat_features.loc[train_mask, feature_cols].values
y_train = users_df.loc[users_df["user_id"].isin(
    flat_features.loc[train_mask, "user_id"]
), "is_ato_victim"].values

X_test = flat_features.loc[test_mask, feature_cols].values
y_test = ground_truth.set_index("user_id").loc[
    flat_features.loc[test_mask, "user_id"]
]["is_ato_victim"].values

# Train XGBoost with class-weight balancing
scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    eval_metric="aucpr",
    random_state=42,
    use_label_encoder=False,
)
xgb_model.fit(X_train, y_train, verbose=False)

# Predict
y_score_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Metrics
roc_auc_xgb = roc_auc_score(y_test, y_score_xgb)
pr_auc_xgb = average_precision_score(y_test, y_score_xgb)

prec_arr_x, rec_arr_x, thresholds_x = precision_recall_curve(y_test, y_score_xgb)
f1_at_95_xgb = 0.0
for p, r, t in zip(prec_arr_x, rec_arr_x, thresholds_x):
    if p >= 0.95 and r > 0:
        f1 = 2 * p * r / (p + r)
        f1_at_95_xgb = max(f1_at_95_xgb, f1)

print("=" * 50)
print("XGBoost ATO Detection Results")
print("=" * 50)
print(f"ROC-AUC:          {roc_auc_xgb:.4f}")
print(f"PR-AUC:           {pr_auc_xgb:.4f}")
print(f"F1 @ 95% Prec:    {f1_at_95_xgb:.4f}")

xgb_metrics = {"ROC-AUC": roc_auc_xgb, "PR-AUC": pr_auc_xgb,
               "F1@95%Prec": f1_at_95_xgb}

## 7. Head-to-Head Comparison: KumoRFM vs XGBoost

We compare the two approaches across metrics that matter to fraud operations teams:

- **ROC-AUC**: Overall discriminative power
- **PR-AUC**: Performance on the imbalanced positive class (critical for fraud)
- **F1 @ 95% Precision**: Recall achievable while maintaining analyst-grade precision (the operating point most fraud teams care about)

In [ ]:
# ==============================================================
# 7.1 Comparison table and visualization
# ==============================================================

if kumo_metrics:
    comparison_data = {
        "Metric": ["ROC-AUC", "PR-AUC", "F1 @ 95% Precision"],
        "KumoRFM": [kumo_metrics["ROC-AUC"], kumo_metrics["PR-AUC"],
                     kumo_metrics["F1@95%Prec"]],
        "XGBoost": [xgb_metrics["ROC-AUC"], xgb_metrics["PR-AUC"],
                     xgb_metrics["F1@95%Prec"]],
    }
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df["Delta"] = comparison_df["KumoRFM"] - comparison_df["XGBoost"]
    comparison_df["Delta%"] = (comparison_df["Delta"] / comparison_df["XGBoost"] * 100).round(1)
    print(comparison_df.to_string(index=False))
    print()

    # Bar chart
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(comparison_data["Metric"]))
    width = 0.3
    bars1 = ax.bar(x - width/2, comparison_data["KumoRFM"], width,
                   label="KumoRFM", color="#FC1373", alpha=0.85)
    bars2 = ax.bar(x + width/2, comparison_data["XGBoost"], width,
                   label="XGBoost", color="steelblue", alpha=0.85)
    ax.set_ylabel("Score")
    ax.set_title("ATO Detection: KumoRFM vs XGBoost")
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_data["Metric"])
    ax.legend()
    ax.set_ylim(0, 1.1)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("KumoRFM metrics not available. Showing XGBoost results only:")
    for k, v in xgb_metrics.items():
        print(f"  {k}: {v:.4f}")
    print("\nRe-run with a valid KumoRFM API key to see the full comparison.")

## 8. Takeover Graph Visualization

Visualizing the relational subgraph around ATO-compromised users reveals the structural patterns that KumoRFM captures. In a real production system, these subgraphs would be used for analyst alert triage: showing *why* a user was flagged, not just *that* they were flagged.

In [ ]:
# ==============================================================
# 8.1 Build and visualize ATO subgraph
# ==============================================================

import networkx as nx

G = nx.Graph()

# Select a few ATO users for visualization
viz_users = ato_user_ids[:6]

for uid in viz_users:
    G.add_node(uid, node_type="user", color="crimson")

    # Add user's sessions and connected devices
    user_sessions = sessions_df[sessions_df["user_id"] == uid]
    for _, sess in user_sessions.iterrows():
        sid = sess["session_id"]
        did = sess["device_id"]
        is_ato = sess["is_ato_session"]

        G.add_node(did, node_type="device",
                   color="orange" if did in attacker_device_pool else "lightblue")
        edge_color = "red" if is_ato else "gray"
        G.add_edge(uid, did, session=sid, ato=is_ato, color=edge_color)

    # Add user's transactions
    user_txns = transactions_df[transactions_df["user_id"] == uid]
    ato_txn_count = user_txns["is_ato_txn"].sum()
    if ato_txn_count > 0:
        txn_node = f"{uid}_txns"
        G.add_node(txn_node, node_type="txn_cluster",
                   color="salmon", label=f"{ato_txn_count} ATO txns")
        G.add_edge(uid, txn_node, color="red")

# Draw
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

pos = nx.spring_layout(G, seed=42, k=2)
node_colors = [G.nodes[n].get("color", "gray") for n in G.nodes()]
edge_colors = [G.edges[e].get("color", "gray") for e in G.edges()]

# Node sizes by type
node_sizes = []
for n in G.nodes():
    ntype = G.nodes[n].get("node_type", "")
    if ntype == "user":
        node_sizes.append(800)
    elif ntype == "device":
        node_sizes.append(400)
    else:
        node_sizes.append(300)

nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                       node_size=node_sizes, alpha=0.8, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors,
                       width=1.5, alpha=0.6, ax=ax)

# Label only users and devices
labels = {}
for n in G.nodes():
    ntype = G.nodes[n].get("node_type", "")
    if ntype in ["user", "device"]:
        labels[n] = n
    elif ntype == "txn_cluster":
        labels[n] = G.nodes[n].get("label", "")
nx.draw_networkx_labels(G, pos, labels, font_size=7, ax=ax)

ax.set_title("ATO Attack Subgraph: Users, Devices, and Fraudulent Activity",
             fontsize=13, fontweight="bold")

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="crimson", label="Compromised User"),
    Patch(facecolor="orange", label="Attacker Device"),
    Patch(facecolor="lightblue", label="Legitimate Device"),
    Patch(facecolor="salmon", label="ATO Transaction Cluster"),
]
ax.legend(handles=legend_elements, loc="upper left", fontsize=9)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Attacker devices shared across users: "
      f"{sum(1 for d in attacker_device_pool if G.has_node(d))}")

In [ ]:
# ==============================================================
# 8.2 XGBoost feature importance (for reference)
# ==============================================================

importances = xgb_model.feature_importances_
feat_imp = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importances
}).sort_values("Importance", ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(feat_imp["Feature"], feat_imp["Importance"], color="steelblue", alpha=0.8)
ax.set_xlabel("Feature Importance (Gain)")
ax.set_title("XGBoost: Top 12 Features for ATO Detection")
plt.tight_layout()
plt.show()

print("\nNote: KumoRFM does not need these hand-engineered features.")
print("It learns equivalent (and richer) representations directly from the graph.")

## 9. Production Deployment Considerations

### Real-Time Scoring Architecture

In production, ATO detection requires **sub-second scoring** at login and transaction time. KumoRFM supports this through:

1. **Graph materialization**: Pre-compute the relational graph from your data warehouse (Snowflake, Databricks) on a schedule.
2. **Online prediction**: Use KumoRFM's online serving endpoint to score individual users in real-time as new sessions/transactions arrive.
3. **Batch + real-time hybrid**: Run batch scoring overnight for risk dashboards, and real-time scoring for transaction-level decisions.

### Integration with Rules Engines

KumoRFM's risk scores should be combined with deterministic rules, not replace them. A practical architecture:

```
[New Session/Transaction]
        |
        v
   [Rules Engine]  --> Hard blocks (known bad IPs, velocity limits)
        |
        v
   [KumoRFM Score] --> Soft score (0-1 probability)
        |
        v
   [Decision Engine] --> Block / Challenge (OTP) / Allow
        |
        v
   [Alert Queue]   --> Analyst review with graph subgraph context
```

### Alert Triage with Graph Context

When KumoRFM flags a user, the relational subgraph (as visualized in Section 8) provides the *explainability* analysts need. Instead of a list of features, they see the structural pattern: which new devices appeared, how sessions cluster, whether loan applications coincide with transaction bursts.

### Monitoring

Track these metrics in production:
- **Precision at operating threshold**: Should stay above 90% to avoid alert fatigue
- **Detection latency**: Time from ATO event to flag
- **False positive rate**: Per-cohort (account type, KYC status) to detect model drift
- **Graph freshness**: Ensure the relational graph reflects recent activity

## 10. Summary

This notebook demonstrated Account Takeover detection using KumoRFM's relational graph approach across three distinct attack patterns (rapid, slow-burn, and cross-product ATO).

**Key takeaways:**

1. **Structural signal matters**: ATO creates relational anomalies (new device-user edges, unusual session-transaction temporal patterns) that flat features can approximate but not fully capture.

2. **No feature engineering required**: KumoRFM operates directly on the multi-table relational graph, eliminating the weeks of feature engineering work that traditional approaches demand.

3. **PR-AUC and high-precision operating points**: These are the metrics that matter for fraud teams. KumoRFM's graph transformer architecture is designed to capture the rare, structurally complex patterns that drive performance at these critical points.

4. **Explainability through graph subgraphs**: Unlike black-box scores, the relational graph provides visual, auditable evidence for analyst review.

### Next in the series

- **Module 3**: Synthetic Identity Fraud Detection (identity fabrication rings across credit bureaus and applications)
- **Module 4**: Money Mule Network Detection (layered fund-flow graphs across accounts and payment rails)

---

*Built by [Abhinav Khare](https://github.com/AbhinavKhareTech) as a community contribution to [kumo-ai/kumo-rfm](https://github.com/kumo-ai/kumo-rfm).*
*For questions or feedback: khare.abhinav@gmail.com*